# Metrics for many classes & many labels

_Metrics & Evaluation — measuring models, data & statistics_

**How to score a model when there are many classes — or when each example can wear several labels at once.**

With two classes you had one confusion matrix and one precision/recall pair. With many classes you get a bigger confusion matrix and a precision/recall pair per class — then you must summarize them into one number (that is what macro / micro / weighted do).

       With multilabel the very notion of "correct" changes: an answer is now a set of labels, so we measure how much two sets overlap (Hamming loss, example-based F1) or how well the model ranks the true labels to the top (LRAP, coverage, ranking loss).

---

This notebook is a practice scaffold for the **Metrics for many classes & many labels** lesson. We build it up one step at a time — first the multiclass digit metrics, then the multilabel set- and ranking-based metrics — so you can see what each number measures before summarizing. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — scikit-learn

### Step 1 — Train a 10-class digit classifier

Multiclass means each example belongs to exactly **one** of many classes. We load the 8x8 handwritten digits (10 classes, 0–9), split off a test set, and fit a logistic-regression classifier. `predict` returns the single best class per example, while `predict_proba` returns a score for every class — we'll need those scores for top-k accuracy.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# 10-class handwritten digits: each 8x8 image flattened to 64 features.
X, y = load_digits(return_X_y=True)

# Stratify keeps the class balance the same in train and test.
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

clf = LogisticRegression(max_iter=5000, random_state=0)
clf.fit(Xtr, ytr)

pred = clf.predict(Xte)          # single best class per example
proba = clf.predict_proba(Xte)   # score for every class (needed for top-k)

### Step 2 — Summarize multiclass performance

With 10 classes you get a precision/recall/F1 **per class**, so you must roll them up. **Top-k accuracy** counts an example correct if the true class is among the model's k highest-scoring classes. The **classification report** prints the per-class scores plus the macro (equal-weight), micro (pooled), and weighted summary rows — each averages the per-class numbers differently.

In [ ]:
from sklearn.metrics import top_k_accuracy_score, classification_report

# Correct if the true class is in the model's top-k highest-scoring classes.
top1 = top_k_accuracy_score(yte, proba, k=1)
top3 = top_k_accuracy_score(yte, proba, k=3)
print("top-1 accuracy:", round(top1, 4))
print("top-3 accuracy:", round(top3, 4))

# Per-class precision/recall/F1 plus macro / micro / weighted summary rows:
print(classification_report(yte, pred, digits=3))

### Step 3 — Score a multilabel problem

Multilabel is different: each example can carry **several** labels at once, so an answer is a *set*. `Y_true` and `Y_pred` are 5 examples × 3 labels of 0/1 memberships, and `Y_score` holds the model's confidences used by the ranking metrics. We measure **set overlap** (Hamming loss, exact-match accuracy, the F1 averages) and **ranking quality** (LRAP, coverage error, ranking loss).

In [ ]:
from sklearn.metrics import (hamming_loss, accuracy_score, f1_score,
    label_ranking_average_precision_score, coverage_error, label_ranking_loss)

# 5 examples, 3 possible labels each (1 = label present).
Y_true = np.array([[1,0,1],[0,1,1],[1,1,0],[0,0,1],[1,0,0]])
Y_pred = np.array([[1,0,1],[0,1,0],[1,1,1],[0,0,1],[0,0,0]])

# Model confidences (not 0/1) — used by the ranking-based metrics:
Y_score = np.array([[0.7,0.9,0.2],[0.3,0.6,0.5],[0.6,0.5,0.7],
                    [0.1,0.2,0.7],[0.4,0.5,0.2]])

# Set-overlap metrics: how close are predicted and true label sets?
print("hamming loss       :", round(hamming_loss(Y_true, Y_pred), 4))
print("subset/exact match :", round(accuracy_score(Y_true, Y_pred), 4))
print("F1 micro           :", round(f1_score(Y_true, Y_pred, average="micro"), 4))
print("F1 macro           :", round(f1_score(Y_true, Y_pred, average="macro"), 4))
print("F1 samples (ex-based):", round(f1_score(Y_true, Y_pred, average="samples"), 4))
print("F1 weighted        :", round(f1_score(Y_true, Y_pred, average="weighted"), 4))

# Ranking metrics: are the true labels scored above the false ones?
print("LRAP               :", round(label_ranking_average_precision_score(Y_true, Y_score), 4))
print("coverage error     :", round(coverage_error(Y_true, Y_score), 4))
print("label ranking loss :", round(label_ranking_loss(Y_true, Y_score), 4))

## Visualize the data & results

_On real 10-class digit data, which class does the model handle worst — and how much does that hide inside a single overall score?_

### Step 1 — Refit and get per-class F1

To see *which* class the model handles worst, we refit the same digit classifier and ask for F1 **per class** with `average=None`. That returns one F1 score for each digit 0–9, instead of collapsing them into a single summary number.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

X, y = load_digits(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

clf = LogisticRegression(max_iter=5000, random_state=0)
clf.fit(Xtr, ytr)
pred = clf.predict(Xte)

# One F1 per digit 0..9 (average=None disables summarizing).
per_class = f1_score(yte, pred, average=None)

### Step 2 — Find the worst class and compare to the summaries

Now find the digit with the lowest F1 and contrast it with the macro and micro summaries. The point: a single overall score can hide a weak class. Macro averages the 10 per-class F1s equally (so the weak class counts), while micro pools all examples (so frequent, easy classes dominate and the weak one nearly vanishes).

In [ ]:
# argmin picks the index (digit) with the smallest F1.
worst = int(np.argmin(per_class))

macro_f1 = f1_score(yte, pred, average="macro")
micro_f1 = f1_score(yte, pred, average="micro")

print("per-class F1:", np.round(per_class, 3))
print("worst class:", worst, "F1=", round(per_class[worst], 3))
print("macro F1   :", round(macro_f1, 3))
print("micro F1   :", round(micro_f1, 3))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A 10-class digit model gets micro-F1 = 0.96 but its per-class F1 for the digit '8' is only 0.92. The leaderboard shows micro-F1. Why might that be hiding a problem, and what should you report?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what micro does: it pools every example's right/wrong counts across all 10 classes into one tally. — _Because big and easy classes contribute most of the pooled counts, a weak class barely moves the micro number._
- Compute the macro-F1 (equal-weight mean of the 10 per-class F1) and read the per-class row. — _Macro gives class '8' a full 1/10 vote, so its weakness shows; the per-class row pinpoints exactly where the model fails._

**Answer:** Micro-F1 = 0.96 is dominated by the many easy digits, so the weak '8' (F1 = 0.92) is invisible in it. Report macro-F1 and the full per-class F1 table; the lowest per-class score (here '8') is the real risk to watch.

</details>

**Problem 2.** In a multilabel tagging task, a model is 90% correct on each individual label, yet its subset (exact-match) accuracy is only ~40%. Is the model broken?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note subset accuracy needs the ENTIRE predicted set to equal the true set for an example to count. — _One wrong label anywhere on an example zeroes the whole row, no partial credit._
- Estimate: if there are ~5 labels and each is right 90% of the time, the chance ALL five are right is about $0.9^5\approx0.59$ — and harder cases push it lower. — _Per-label accuracy and exact-match measure different things; high per-label accuracy can still give low exact-match._

**Answer:** Not necessarily. Exact-match is brutally strict, so strong per-label accuracy still yields modest subset accuracy. Report Hamming loss (here ~0.10) and example-based F1 alongside it for a fair picture.

</details>

**Problem 3.** For a label-ranking model you get LRAP = 0.73, coverage error = 2.2, and label ranking loss = 0.5. In plain words, what do these say?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read LRAP: average over true labels of (fraction of labels ranked at or above it that are also true). 1.0 = all true labels sit at the very top. — _0.73 means true labels are usually near the top but some wrong labels slip above them._
- Read coverage error and ranking loss: how far down to gather all true labels, and the fraction of (true, false) label pairs that are mis-ordered. — _Coverage 2.2 = scan about 2.2 labels deep; ranking loss 0.5 = half the right/wrong pairs are in the wrong order._

**Answer:** True labels usually rank high (LRAP 0.73) but not perfectly; you must scan ~2.2 labels down to catch all true ones (coverage 2.2), and half of the true-vs-false label pairs are mis-ordered (ranking loss 0.5). Lower coverage and ranking loss, higher LRAP, are better.

</details>